In [18]:
# Loan Eligibility Model Script
# This script loads training and test datasets, preprocesses the data, trains a RandomForestClassifier,
# evaluates its performance, predicts loan eligibility on the test set, saves the trained model,
# and provides a realistic dummy template for new input.

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib  # for saving the model

# 1. Load the data
train = pd.read_csv('train_data.csv')  # adjust path if needed
test = pd.read_csv('test_data.csv')    # adjust path if needed

# 1a. Convert LoanAmount from 'thousands' to actual rupees to match income scales
train['LoanAmount'] = train['LoanAmount'] * 1000
test['LoanAmount'] = test['LoanAmount'] * 1000

# 2. Define feature groups
numeric_features = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']
categorical_features = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']

# 3. Build preprocessing pipelines
numeric_transformer = SimpleImputer(strategy='median')

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# 4. Create modeling pipeline
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# 5. Prepare training data
X = train.drop(['Loan_ID', 'Loan_Status'], axis=1)
y = train['Loan_Status'].map({'Y': 1, 'N': 0})

# 6. Split into train/validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 7. Train the model
clf.fit(X_train, y_train)

# 8. Evaluate on validation set
y_pred_val = clf.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_val, y_pred_val))
print(classification_report(y_val, y_pred_val))

# 9. Predict on test set
X_test = test.drop('Loan_ID', axis=1)
test_preds = clf.predict(X_test)
test['Loan_Status_Prediction'] = ['Y' if pred == 1 else 'N' for pred in test_preds]
print(test[['Loan_ID', 'Loan_Status_Prediction']].head())

# 10. Create a realistic dummy/template dataframe for future inputs
# LoanAmount now expects actual rupee values (e.g., 120000 instead of 120)
dummy_template = pd.DataFrame([
    {
        'Gender': 'Male', 'Married': 'Yes', 'Dependents': '0', 'Education': 'Graduate',
        'Self_Employed': 'No', 'ApplicantIncome': 5000, 'CoapplicantIncome': 0,
        'LoanAmount': 120000, 'Loan_Amount_Term': 360, 'Credit_History': 1.0, 'Property_Area': 'Urban'
    },
    {
        'Gender': 'Female', 'Married': 'No', 'Dependents': '1', 'Education': 'Not Graduate',
        'Self_Employed': 'Yes', 'ApplicantIncome': 3000, 'CoapplicantIncome': 1500,
        'LoanAmount': 100000, 'Loan_Amount_Term': 360, 'Credit_History': 0.0, 'Property_Area': 'Rural'
    },
    {
        'Gender': 'Male', 'Married': 'Yes', 'Dependents': '2', 'Education': 'Graduate',
        'Self_Employed': 'No', 'ApplicantIncome': 6500, 'CoapplicantIncome': 2000,
        'LoanAmount': 200000, 'Loan_Amount_Term': 360, 'Credit_History': 1.0, 'Property_Area': 'Semiurban'
    },
    {
        'Gender': 'Female', 'Married': 'Yes', 'Dependents': '3+', 'Education': 'Graduate',
        'Self_Employed': 'No', 'ApplicantIncome': 4000, 'CoapplicantIncome': 500,
        'LoanAmount': 150000, 'Loan_Amount_Term': 180, 'Credit_History': 1.0, 'Property_Area': 'Urban'
    },
    {
        'Gender': 'Male', 'Married': 'No', 'Dependents': '0', 'Education': 'Not Graduate',
        'Self_Employed': 'Yes', 'ApplicantIncome': 2500, 'CoapplicantIncome': 0,
        'LoanAmount': 80000, 'Loan_Amount_Term': 360, 'Credit_History': 0.0, 'Property_Area': 'Semiurban'
    }
])

print("\nRealistic Dummy Input Template (actual rupees):")
print(dummy_template)

# 11. Function to predict new applicants
def predict_eligibility(df_new):
    """
    Predict loan eligibility (Y/N) for new applicant DataFrame.
    """
    preds = clf.predict(df_new)
    return ['Y' if p == 1 else 'N' for p in preds]

# 12. Save the trained model
joblib.dump(clf, 'loan_model9.pkl')
print("Model saved to loan_model9.pkl")





Validation Accuracy: 0.7642276422764228
              precision    recall  f1-score   support

           0       0.79      0.44      0.57        43
           1       0.76      0.94      0.84        80

    accuracy                           0.76       123
   macro avg       0.77      0.69      0.70       123
weighted avg       0.77      0.76      0.74       123

    Loan_ID Loan_Status_Prediction
0  LP001015                      Y
1  LP001022                      Y
2  LP001031                      Y
3  LP001035                      Y
4  LP001051                      N

Realistic Dummy Input Template (actual rupees):
   Gender Married Dependents     Education Self_Employed  ApplicantIncome  \
0    Male     Yes          0      Graduate            No             5000   
1  Female      No          1  Not Graduate           Yes             3000   
2    Male     Yes          2      Graduate            No             6500   
3  Female     Yes         3+      Graduate            No          

In [19]:
print(predict_eligibility(dummy_template))

['Y', 'N', 'Y', 'Y', 'N']
